In [ ]:
# !pip install --upgrade pip
# !pip install pandas numpy matplotlib scikit-learn

## Importações de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# PCA — redução de dimensionalidade
from sklearn.decomposition import PCA

# Padronização de escala (pré-requisito do PCA)
from sklearn.preprocessing import StandardScaler

# Dataset Iris: clássico da literatura de ML, 150 amostras, 4 variáveis numéricas
from sklearn.datasets import load_iris

---
## Módulo 01

### O que é PCA?

**PCA (Principal Component Analysis)** é uma técnica de **redução de dimensionalidade** que transforma um conjunto de variáveis correlacionadas em um conjunto menor de variáveis não correlacionadas chamadas **componentes principais**.

Cada componente principal é uma combinação linear das variáveis originais, ordenadas de forma que:
- **PC1** captura a maior variância possível dos dados
- **PC2** captura a segunda maior variância, ortogonal à PC1
- **PC3**, **PC4**... e assim por diante

**Quando usar PCA?**
- Quando o dataset tem muitas variáveis numéricas correlacionadas
- Para visualizar dados de alta dimensionalidade em 2D ou 3D
- Para remover multicolinearidade antes de aplicar modelos preditivos
- Para compressão de dados sem perda significativa de informação

**Pré-requisito obrigatório:** as variáveis devem estar na mesma escala — por isso o `StandardScaler` é sempre aplicado antes do PCA.

### Dataset — Iris

O **Iris** é um dos datasets mais clássicos de Machine Learning.

| Coluna | Tipo | Descrição |
|---|---|---|
| sepal length (cm) | numérica | comprimento da sépala |
| sepal width (cm) | numérica | largura da sépala |
| petal length (cm) | numérica | comprimento da pétala |
| petal width (cm) | numérica | largura da pétala |
| species | categórica | setosa / versicolor / virginica |

- **150 amostras**, **4 features numéricas**, **3 classes**

In [ ]:
# Carregar o dataset Iris via scikit-learn
iris = load_iris()

# Converter para DataFrame pandas
# iris.data       → matriz NumPy com as 4 features
# iris.feature_names → nomes originais das colunas
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

# Adicionar a coluna de espécie (alvo)
# iris.target       → array de inteiros: 0, 1, 2
# iris.target_names → array de strings: ['setosa', 'versicolor', 'virginica']
df_iris['species'] = iris.target_names[iris.target]

df_iris.head()

In [ ]:
print(df_iris.shape)   # (linhas, colunas)

In [ ]:
print(df_iris.dtypes)

In [ ]:
# Estatísticas descritivas — note as escalas diferentes entre as features
df_iris.describe().round(2)

In [ ]:
# Distribuição das espécies — dataset balanceado
df_iris['species'].value_counts()

---
## Módulo 02

### Padronização — `StandardScaler`

O PCA é sensível à escala das variáveis. Se uma variável tem valores na casa dos milhares e outra na casa das unidades, a primeira vai dominar os componentes principais — não por ser mais importante, mas apenas por ter valores maiores.

O `StandardScaler` transforma cada variável para ter:
- **Média = 0**
- **Desvio padrão = 1**

$$z = \frac{x - \mu}{\sigma}$$

> ⚠️ **Regra de ouro:** nunca aplique PCA sem padronizar os dados antes.

In [ ]:
# Separar apenas as features numéricas (remover a coluna de espécie)
X = df_iris.drop('species', axis=1)

print('Features utilizadas:')
print(X.columns.tolist())

In [ ]:
# Instanciar e ajustar o StandardScaler
# fit_transform() = aprende média e desvio padrão + transforma em uma única chamada
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Resultado é um array NumPy — converter para DataFrame facilita a inspeção
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled.head()

In [ ]:
# Verificar: média ≈ 0 e desvio padrão ≈ 1 para todas as colunas
print('Média após padronização (deve ser ≈ 0):')
print(df_scaled.mean().round(10))
print()
print('Desvio padrão após padronização (deve ser ≈ 1):')
print(df_scaled.std().round(2))

---
## Módulo 03

### PCA Completo — Análise de Variância Explicada

Antes de decidir quantos componentes manter, rodamos o PCA completo (sem reduzir dimensões) para entender **quanto cada componente explica da variância total** dos dados.

Conceitos-chave:
- **`explained_variance_ratio_`** → proporção da variância explicada por cada componente
- **Variância acumulada** → soma progressiva das proporções
- **Critério prático:** manter o menor número de componentes que explique ≥ 95% da variância

In [ ]:
# PCA sem restringir o número de componentes
# Como temos 4 features, teremos no máximo 4 componentes principais
pca_full = PCA()
pca_full.fit(X_scaled)

In [ ]:
# Variância explicada por cada componente (em %)
variancia_por_componente = pca_full.explained_variance_ratio_ * 100

for i, v in enumerate(variancia_por_componente, start=1):
    print(f'PC{i}: {v:.2f}%')

In [ ]:
# Variância acumulada — quanto já explicamos ao adicionar cada componente
variancia_acumulada = np.cumsum(pca_full.explained_variance_ratio_) * 100

for i, v in enumerate(variancia_acumulada, start=1):
    print(f'{i} componente(s): {v:.2f}% da variância total')

#### Scree Plot — gráfico de cotovelo

O **Scree Plot** (ou gráfico de cotovelo) mostra a variância acumulada em função do número de componentes.
A linha tracejada em 95% é o limiar mais comum na prática:
escolhemos o menor número de componentes que cruza essa linha.

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, 5),
    variancia_acumulada,
    marker='o',
    color='steelblue',
    linewidth=2
)

# Linha de referência em 95%
plt.axhline(y=95, linestyle='--', color='red', label='95% de variância')

# Anotar os valores em cada ponto
for i, v in enumerate(variancia_acumulada):
    plt.annotate(f'{v:.1f}%', (i + 1, v), textcoords='offset points',
                 xytext=(8, -12), fontsize=9)

plt.xlabel('Número de Componentes')
plt.ylabel('Variância Acumulada (%)')
plt.title('Scree Plot — Variância Acumulada por Componente')
plt.xticks(range(1, 5))
plt.ylim(0, 105)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Interpretação automática: quantos componentes para atingir 95%?
n_componentes_95 = np.argmax(variancia_acumulada >= 95) + 1
print(f'Componentes necessários para ≥ 95% de variância: {n_componentes_95}')
print(f'Variância explicada com {n_componentes_95} componente(s): {variancia_acumulada[n_componentes_95 - 1]:.2f}%')

---
## Módulo 04

### PCA com 2 Componentes — Redução de Dimensionalidade

Com base no Scree Plot, escolhemos **2 componentes** para a visualização.

Isso significa que estamos **comprimindo 4 variáveis em 2**, mantendo a maior parte da estrutura dos dados.
Cada linha do dataset original — antes um ponto em um espaço de 4 dimensões — passa a ser um ponto em 2D.

In [ ]:
# PCA restrito a 2 componentes
pca2 = PCA(n_components=2)

# fit_transform() = aprende os componentes + projeta os dados
# X_pca é um array de shape (150, 2)
X_pca = pca2.fit_transform(X_scaled)

print(f'Shape original:  {X_scaled.shape}')   # (150, 4)
print(f'Shape após PCA:  {X_pca.shape}')       # (150, 2)

In [ ]:
# Variância explicada pelos 2 componentes selecionados
print(f'PC1 explica: {pca2.explained_variance_ratio_[0]*100:.1f}%')
print(f'PC2 explica: {pca2.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total:       {pca2.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# Montar DataFrame com os scores dos 2 componentes + espécie
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['species'] = df_iris['species']

df_pca.head()

#### Loadings — como cada variável original contribui para os componentes

Os **loadings** (ou autovetores) indicam o peso de cada variável original em cada componente principal.
Um loading alto (positivo ou negativo) significa que aquela variável tem grande influência naquele componente.

In [ ]:
# components_ → matriz de shape (n_components, n_features)
loadings = pd.DataFrame(
    pca2.components_,
    columns=X.columns,
    index=['PC1', 'PC2']
)

loadings.round(3)

In [ ]:
# Visualizar os loadings como gráfico de barras
loadings.T.plot(kind='bar', figsize=(8, 4), edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Loadings — Contribuição das Variáveis Originais por Componente')
plt.ylabel('Loading')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## Módulo 05

### Visualização 2D — Projeção PCA com Rótulo de Espécie

Com os dados projetados em 2 dimensões, podemos criar um scatter plot colorido por espécie.
Se o PCA capturou bem a estrutura dos dados, grupos naturais (classes) devem aparecer **separados visualmente** — mesmo que o PCA não tenha usado a coluna `species` em nenhum momento.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Plotar cada espécie separadamente para controlar cor e rótulo
cores = {'setosa': 'steelblue', 'versicolor': 'darkorange', 'virginica': 'seagreen'}

for especie, grupo in df_pca.groupby('species'):
    ax.scatter(
        grupo['PC1'],
        grupo['PC2'],
        label=especie,
        color=cores[especie],
        alpha=0.7,
        edgecolors='white',
        s=60
    )

# Percentual de variância nos eixos
pc1_var = pca2.explained_variance_ratio_[0] * 100
pc2_var = pca2.explained_variance_ratio_[1] * 100

ax.set_xlabel(f'PC1 ({pc1_var:.1f}% da variância)')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}% da variância)')
ax.set_title('PCA — Iris Dataset (2 Componentes)')
ax.legend(title='Espécie')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Centróides de cada espécie no espaço PCA
# Ajuda a entender o posicionamento médio de cada grupo
df_pca.groupby('species')[['PC1', 'PC2']].mean().round(3)

---
## Módulo 06

### Biplot — Variáveis e Amostras no Mesmo Gráfico

O **biplot** combina o scatter plot das amostras com as setas dos loadings.
Permite ver simultaneamente:
- Como as amostras se distribuem no espaço PCA
- Quais variáveis originais mais contribuem para cada componente
- A direção e intensidade de cada variável

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Scatter das amostras
for especie, grupo in df_pca.groupby('species'):
    ax.scatter(grupo['PC1'], grupo['PC2'],
               label=especie, color=cores[especie], alpha=0.5, s=40, edgecolors='white')

# Setas dos loadings (multiplicadas por fator de escala para visibilidade)
fator = 3
for i, feature in enumerate(X.columns):
    ax.annotate(
        '',
        xy=(pca2.components_[0, i] * fator, pca2.components_[1, i] * fator),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
    )
    ax.text(
        pca2.components_[0, i] * fator * 1.15,
        pca2.components_[1, i] * fator * 1.15,
        feature.replace(' (cm)', ''),
        fontsize=9, ha='center', color='darkred'
    )

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)')
ax.set_title('Biplot — Amostras + Loadings das Variáveis')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.legend(title='Espécie')
plt.tight_layout()
plt.show()

---
## Resumo Geral

| Etapa | Código | O que faz |
|---|---|---|
| Carregar dataset | `load_iris()` | Retorna dict com dados, rótulos e nomes |
| Padronizar | `StandardScaler().fit_transform(X)` | Média=0, DP=1 em todas as features |
| PCA completo | `PCA().fit(X_scaled)` | Calcula todos os componentes |
| Variância explicada | `explained_variance_ratio_` | Proporção de variância por componente |
| Scree Plot | `np.cumsum(explained_variance_ratio_)` | Variância acumulada — critério de corte |
| PCA reduzido | `PCA(n_components=2).fit_transform(X)` | Projeta dados em 2D |
| Loadings | `pca.components_` | Peso de cada variável original por PC |
| Visualização | `scatter(PC1, PC2)` | Grupos naturais visíveis em 2D |
| Biplot | scatter + setas de loadings | Amostras e variáveis no mesmo espaço |